## Executive Summary

**Purpose:** Model the static hysteresis loop of soft magnetic steel from the data a steel datasheet provides.

**What it does:** Implements the Takács model in its basic form: closed-form ascending and descending loop branches, their exact inverses (H from B), remanence, coercivity and loop energy, and a fit of the three parameters to datasheet data. `DynamicLoopModel` widens the static loop for AC operation with Bertotti's classical-eddy and excess fields, using the coefficients of `fit_bertotti`.

### Why It Matters
A single-valued B–H curve gives the magnetising current but no hysteresis. With a loop model, a voltage-driven device such as a transformer gets its no-load current waveform (peaks, harmonics, the in-phase loss component) analytically from B(t).

### Quick Start
**Inputs:** points on the normal magnetisation curve (loop tips) and the static hysteresis energy per cycle at several peak flux densities (from the Bertotti loss separation).

**Outputs:** `TakacsModel` with loops, H(t) for a given B(t), remanence, coercivity and loop energy; `DynamicLoopModel` with H(t) and the energy per cycle at a given frequency.

### Limits of the basic form
- The dynamic widening assumes uniform B through the sheet thickness (no skin effect): valid at mains frequency, not in the kHz range.
- Symmetric loops only: no DC bias and no minor loops inside a half cycle.
- The normal curve has a fixed tanh shape; a steel with a sharp knee and a slow approach to saturation may not fit well.

## How It Fits Into the Motor Design Workflow

**Lives in:** `emachines.magnetics`, next to the B–H models, so transformers, solenoids and motors can share it.

**Upstream:** the static loop energy comes from the hysteresis term of `fit_bertotti` (`emachines.magnetics.iron_loss`); the dynamic widening uses its eddy and excess terms.

**Downstream:** the planned transformer module (no-load current waveform).

In [ ]:
#| default_exp magnetics.hysteresis
#| export
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from scipy.optimize import brentq, least_squares

In [ ]:
#| hide
import matplotlib.pyplot as plt
from scipy.integrate import quad

## Theory: the Takács Model

In normalised units $x = H/h_0$ and $y = B/B_s$, a symmetric loop with its tip at $x_m$ has two branches (Takács, 2001):

$$\text{ascending:}\quad y = \tanh(x - a_0) + b, \qquad \text{descending:}\quad y = \tanh(x + a_0) - b$$

$$b = \tfrac{1}{2}\left[\tanh(x_m + a_0) - \tanh(x_m - a_0)\right]$$

$a_0$ shifts the branches apart and so creates the hysteresis; $b$ is not a free parameter, it closes the loop at its tip. The three material parameters are $B_s$ [T], $h_0$ [A/m] and $a_0$ [–].

Everything else follows in closed form:

| Quantity | Normalised form |
|---|---|
| Loop tip (normal magnetisation curve) | $y_m = \tfrac{1}{2}\left[\tanh(x_m + a_0) + \tanh(x_m - a_0)\right]$ |
| Remanence (descending branch at $x = 0$) | $y_r = \tanh(a_0) - b$ |
| Coercivity (ascending branch at $y = 0$) | $x_c = a_0 - \operatorname{artanh}(b)$ |
| Loop area | $A = 2\ln\dfrac{\cosh(x_m + a_0)}{\cosh(x_m - a_0)} - 2x_m\left[\tanh(x_m + a_0) - \tanh(x_m - a_0)\right]$ |
| Inverse, rising $B$ | $x = \operatorname{artanh}(y - b) + a_0$ |
| Inverse, falling $B$ | $x = \operatorname{artanh}(y + b) - a_0$ |

The loop area follows from integrating descending minus ascending branch over $[-x_m, x_m]$. In physical units the energy per cycle is $W = B_s h_0 A$ [J/m³].

The inverses are valid on the whole loop: on the ascending branch $y - b$ runs from $-\tanh(x_m + a_0)$ to $\tanh(x_m - a_0)$, both inside $(-1, 1)$, and likewise for the descending branch. So for a voltage-driven device, where $B(t)$ is known, $H(t)$ is explicit.

The tip curve $y_m(x_m)$ rises monotonically from 0 to 1 (its slope $\tfrac12[\operatorname{sech}^2(x_m + a_0) + \operatorname{sech}^2(x_m - a_0)]$ is positive), so every $0 < B_m < B_s$ has exactly one loop.

$B_s$ and $h_0$ are fitted scales, not physical constants: $B_s$ is not the saturation polarisation $J_s$.

In [ ]:
#| export
def _logcosh(z):
    """ln(cosh(z)) without overflow for large |z|."""
    z = np.abs(z)
    return z + np.log1p(np.exp(-2 * z)) - np.log(2.0)


@dataclass(frozen=True)
class TakacsModel:
    """Takács hysteresis model, basic form: static, symmetric loops.

    Branches in normalised units x = H/h0, y = B/B_s, for a loop with tip at x_m:
    ascending y = tanh(x - a0) + b, descending y = tanh(x + a0) - b,
    with b = (tanh(x_m + a0) - tanh(x_m - a0)) / 2.
    """
    B_s: float  # [T] flux-density scale (a fitted value, not the saturation polarisation)
    h0: float  # [A/m] field scale
    a0: float  # [-] coercive field of the major loop, in units of h0

    def __post_init__(self):
        if not (self.B_s > 0 and self.h0 > 0 and self.a0 > 0):
            raise ValueError(f"B_s, h0 and a0 must be positive, got {self.B_s}, {self.h0}, {self.a0}")

    def _x_m(self, H_m):
        H_m = np.asarray(H_m, dtype=float)
        if np.any(H_m <= 0):
            raise ValueError("loop tip field H_m must be positive")
        return H_m / self.h0

    def _b(self, x_m):
        return 0.5 * (np.tanh(x_m + self.a0) - np.tanh(x_m - self.a0))

    def tip_B(self, H_m):
        """Loop tip flux density [T] for tip field H_m [A/m]: the normal magnetisation curve."""
        x_m = self._x_m(H_m)
        return self.B_s * 0.5 * (np.tanh(x_m + self.a0) + np.tanh(x_m - self.a0))

    def tip_H(self, B_m):
        """Tip field H_m [A/m] of the loop whose tip is at B_m [T], with 0 < B_m < B_s."""
        B_m = np.asarray(B_m, dtype=float)
        if np.any(B_m <= 0) or np.any(B_m >= self.B_s):
            raise ValueError(f"B_m must be in (0, B_s) = (0, {self.B_s}) T")
        def solve(y):
            f = lambda x: 0.5 * (np.tanh(x + self.a0) + np.tanh(x - self.a0)) - y
            hi = self.a0 + 1.0
            while f(hi) < 0:
                hi *= 2
            return brentq(f, 0.0, hi, xtol=1e-14)
        x = np.vectorize(solve)(B_m / self.B_s)
        return self.h0 * x if x.ndim else float(self.h0 * x)

    def normal_H(self, B):
        """H [A/m] on the normal magnetisation curve: single-valued, no hysteresis.

        Odd extension of the loop-tip curve, H(-B) = -H(B) and H(0) = 0, for |B| < B_s.
        """
        B = np.asarray(B, dtype=float)
        H = np.zeros_like(B)
        nz = B != 0
        if np.any(nz):
            H[nz] = np.sign(B[nz]) * np.atleast_1d(self.tip_H(np.abs(B[nz])))
        return H if H.ndim else float(H)

    def branch_B(self, H, H_m, rising: bool):
        """B [T] on the ascending (rising=True) or descending branch of the loop with tip H_m."""
        x_m = self._x_m(H_m)
        x = np.asarray(H, dtype=float) / self.h0
        b = self._b(x_m)
        y = np.tanh(x - self.a0) + b if rising else np.tanh(x + self.a0) - b
        return self.B_s * y

    def branch_H(self, B, B_m, rising: bool):
        """H [A/m] on the ascending (rising=True) or descending branch of the loop with tip B_m.

        Exact inverse of `branch_B`; B is clipped to [-B_m, B_m].
        """
        x_m = self.tip_H(B_m) / self.h0
        b = self._b(x_m)
        y = np.clip(np.asarray(B, dtype=float), -B_m, B_m) / self.B_s
        x = np.arctanh(y - b) + self.a0 if rising else np.arctanh(y + b) - self.a0
        return self.h0 * x

    def loop(self, H_m, n: int = 200):
        """Closed loop with tip H_m: (H, B) arrays, ascending from -H_m to H_m, then descending back."""
        H_up = np.linspace(-H_m, H_m, n)
        H = np.concatenate([H_up, H_up[::-1]])
        B = np.concatenate([self.branch_B(H_up, H_m, True), self.branch_B(H_up[::-1], H_m, False)])
        return H, B

    def remanence(self, H_m):
        """Remanent flux density [T] of the loop with tip H_m (descending branch at H = 0)."""
        return self.B_s * (np.tanh(self.a0) - self._b(self._x_m(H_m)))

    def coercivity(self, H_m):
        """Coercive field [A/m] of the loop with tip H_m (ascending branch at B = 0)."""
        return self.h0 * (self.a0 - np.arctanh(self._b(self._x_m(H_m))))

    def loop_energy(self, H_m):
        """Hysteresis energy per cycle [J/m³] of the loop with tip H_m (the loop area)."""
        x_m, a0 = self._x_m(H_m), self.a0
        area = (2 * (_logcosh(x_m + a0) - _logcosh(x_m - a0))
                - 2 * x_m * (np.tanh(x_m + a0) - np.tanh(x_m - a0)))
        return self.B_s * self.h0 * area

    def field_waveform(self, B):
        """H(t) [A/m] for one period of B(t) [T] sampled at equal time steps.

        The loop tip is max|B|; each sample takes the ascending branch where B is
        rising and the descending branch where it is falling. The basic form covers
        symmetric loops only, so B(t) must have equal positive and negative peaks
        and exactly one maximum and one minimum per period.
        """
        B = np.asarray(B, dtype=float)
        B_m = np.max(np.abs(B))
        if not np.isclose(B.max(), -B.min(), rtol=1e-3):
            raise ValueError("B(t) peaks are not symmetric; biased loops are not in the basic Takács form")
        dB = np.roll(B, -1) - np.roll(B, 1)            # central difference, periodic
        s = np.sign(dB[dB != 0])
        if np.count_nonzero(s != np.roll(s, 1)) != 2:
            raise ValueError("B(t) has minor reversals; minor loops are not in the basic Takács form")
        rising = dB > 0
        H = np.empty_like(B)
        H[rising] = self.branch_H(B[rising], B_m, True)
        H[~rising] = self.branch_H(B[~rising], B_m, False)
        return H

In [ ]:
# Examples
# Illustrative parameters only — not fitted to any real steel.
m = TakacsModel(B_s=1.6, h0=50.0, a0=0.8)
H_m = 2.5 * m.h0                                     # x_m = 2.5
x_m, a0 = 2.5, 0.8
b = 0.5 * (np.tanh(x_m + a0) - np.tanh(x_m - a0))
assert np.isclose(m.tip_B(H_m), m.branch_B(H_m, H_m, rising=True))      # branches meet at the tip
assert np.isclose(m.tip_B(H_m), m.branch_B(H_m, H_m, rising=False))
assert np.isclose(m.remanence(H_m), 1.6 * (np.tanh(a0) - b))
assert np.isclose(m.branch_B(m.coercivity(H_m), H_m, rising=True), 0, atol=1e-12)
assert np.isclose(m.tip_H(m.tip_B(H_m)), H_m)
assert np.isclose(m.normal_H(-m.tip_B(H_m)), -H_m) and m.normal_H(0.0) == 0.0   # odd, single-valued
print(f"tip B = {m.tip_B(H_m):.4f} T, B_r = {m.remanence(H_m):.4f} T, "
      f"H_c = {m.coercivity(H_m):.2f} A/m, W = {m.loop_energy(H_m):.1f} J/m³")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for xm in [1.0, 1.5, 2.5, 5.0]:
    H, B = m.loop(xm * m.h0)
    ax.plot(H, B, lw=1.4, label=f"tip H = {xm * m.h0:.0f} A/m")
H_n = np.linspace(1, 5 * m.h0, 200)
ax.plot(H_n, m.tip_B(H_n), "k--", lw=1, label="normal curve (loop tips)")
ax.set_xlabel("H (A/m)"); ax.set_ylabel("B (T)")
ax.set_title("Takács loops, illustrative parameters")
ax.grid(True, alpha=0.4); ax.legend(fontsize=8)
plt.show()

## Field Waveform for a Given B(t)

For a sinusoidal supply the flux density is known, $B(t) = \hat B \sin\omega t$, and the exact inverses give $H(t)$ point by point. The magnetising current of a winding with $N$ turns on a core of path length $l_c$ is then $i(t) = H(t)\, l_c / N$.

Because the loop is static, this is the current at very low frequency. At 50 Hz eddy currents widen the loop: see *Dynamic Widening* below.

In [ ]:
# Examples
theta = np.linspace(0, 2 * np.pi, 720, endpoint=False)
B_t = 1.3 * np.sin(theta)
H_t = m.field_waveform(B_t)
# H(t) retraces the loop: rising samples lie on the ascending branch
up = np.roll(B_t, -1) > np.roll(B_t, 1)
assert np.allclose(m.branch_B(H_t[up], H_t.max(), rising=True), B_t[up])
# half-wave symmetry: H(t + T/2) = -H(t), so only odd harmonics
assert np.allclose(H_t[360:], -H_t[:360])
c = np.fft.rfft(H_t) / len(H_t) * 2
print("harmonic amplitudes of H (A/m), orders 1-7:", np.round(np.abs(c[1:8]), 2))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(np.degrees(theta), H_t, label="H(t)")
ax2 = ax.twinx(); ax2.plot(np.degrees(theta), B_t, "k:", label="B(t)")
ax.set_xlabel("ωt (deg)"); ax.set_ylabel("H (A/m)"); ax2.set_ylabel("B (T)")
ax.set_title("Field for sinusoidal B, illustrative parameters"); ax.grid(True, alpha=0.4)
plt.show()

## Fitting to Datasheet Data

A datasheet gives two things about each symmetric loop:

1. **The tip**, a point $(\hat H, \hat B)$ on the normal magnetisation curve.
2. **The area**, the static hysteresis energy per cycle. From the Bertotti separation $p_h = k_h f \hat B^{2}$ [W/kg] (`fit_bertotti`, default exponent 2), the energy per cycle is $W_h = k_h \hat B^{2} \rho$ [J/m³] with $\rho$ the density.

`fit_takacs` finds $B_s$, $h_0$ and $a_0$ by least squares on the relative errors of both. There are more data points than parameters, so the residuals also show whether the basic form suits the material. Remanence and coercivity are then predictions of the fitted model; without a measured loop they cannot be checked.

In [ ]:
#| export
def fit_takacs(H_tip, B_tip, B_loop, W_loop) -> dict:
    """Fit TakacsModel to loop tips and loop energies.

    Args:
        H_tip, B_tip: points on the normal magnetisation curve [A/m], [T].
        B_loop: peak flux densities [T] at which the loop energy is known.
        W_loop: static hysteresis energy per cycle at B_loop [J/m³].

    Returns:
        Dict with 'B_s', 'h0', 'a0', the fitted 'takacs' model, and the relative
        rms errors 'rel_rms_tip' and 'rel_rms_energy'; 'model' is "Takács".
    """
    H_tip, B_tip = np.asarray(H_tip, float), np.asarray(B_tip, float)
    B_loop, W_loop = np.asarray(B_loop, float), np.asarray(W_loop, float)
    if len(H_tip) != len(B_tip) or len(B_loop) != len(W_loop):
        raise ValueError("H_tip/B_tip and B_loop/W_loop must have equal lengths")
    if len(H_tip) + len(W_loop) < 3 or len(W_loop) == 0:
        raise ValueError("need at least one loop energy and three data points in total")
    for name, arr in [("H_tip", H_tip), ("B_tip", B_tip), ("B_loop", B_loop), ("W_loop", W_loop)]:
        if np.any(arr <= 0):
            raise ValueError(f"{name} must be positive")
    B_max = max(B_tip.max(), B_loop.max())

    def residuals(p):
        mdl = TakacsModel(*np.exp(p))
        r_tip = mdl.tip_B(H_tip) / B_tip - 1
        r_w = mdl.loop_energy(mdl.tip_H(B_loop)) / W_loop - 1
        return np.concatenate([r_tip, r_w])

    B_s0 = 1.2 * B_max
    h00 = float(np.interp(0.5 * B_s0, np.sort(B_tip), H_tip[np.argsort(B_tip)]))
    lo = [np.log(1.0001 * B_loop.max()), -np.inf, -np.inf]   # tip_H needs B_loop < B_s
    sol = least_squares(residuals, np.log([B_s0, h00, 0.5]), bounds=(lo, np.inf),
                        x_scale="jac", xtol=1e-14, ftol=1e-14)
    B_s, h0, a0 = np.exp(sol.x)
    r = residuals(sol.x)
    n = len(H_tip)
    return {"B_s": B_s, "h0": h0, "a0": a0, "takacs": TakacsModel(B_s, h0, a0),
            "rel_rms_tip": float(np.sqrt(np.mean(r[:n] ** 2))),
            "rel_rms_energy": float(np.sqrt(np.mean(r[n:] ** 2))), "model": "Takács"}

In [ ]:
# Examples
# Self-consistency check: data generated from a known model are fitted back.
# The data are synthetic, made by TakacsModel itself — not datasheet values.
true = TakacsModel(B_s=1.6, h0=50.0, a0=0.8)
H_tip = np.array([40, 60, 80, 120, 200, 400])
B_loop = np.array([0.5, 0.8, 1.0, 1.2, 1.4])
fit = fit_takacs(H_tip, true.tip_B(H_tip), B_loop, true.loop_energy(true.tip_H(B_loop)))
assert np.allclose([fit["B_s"], fit["h0"], fit["a0"]], [1.6, 50.0, 0.8], rtol=1e-6)
assert fit["rel_rms_tip"] < 1e-9 and fit["rel_rms_energy"] < 1e-9

## Dynamic Widening: Bertotti's Field Separation

At AC the field needed to drive $B(t)$ through a lamination is the static hysteresis field plus two rate-dependent terms (Bertotti, 1988):

$$H(t) = H_\text{Takács}(B) + c_{cl}\,\dot B + c_{exc}\,\operatorname{sign}(\dot B)\,|\dot B|^{1/2}$$

| Term | Origin | Loss |
|---|---|---|
| $H_\text{Takács}(B)$ | static domain-wall pinning | $p_h$, the static loop area |
| $c_{cl}\,\dot B$ | classical eddy currents; physically $c_{cl} = \sigma d^2/12$ | $p_e \propto f^2\hat B^2$ |
| $c_{exc}\operatorname{sign}(\dot B)|\dot B|^{1/2}$ | excess (domain-wall) eddy currents | $p_a \propto f^{1.5}\hat B^{1.5}$ |

At the loop tip $\dot B = 0$, so the tip does not move; the loop widens where $B$ crosses zero.

**Coefficients from `fit_bertotti`.** Integrating $H\,dB$ over one sinusoidal cycle and equating with $p_e = k_e (f\hat B)^2$ and $p_a = k_a (f\hat B)^{1.5}$ [W/kg]:

$$c_{cl} = \frac{k_e\,\rho}{2\pi^2}, \qquad c_{exc} = \frac{k_a\,\rho}{C\sqrt{2\pi}}, \qquad C = \int_0^{2\pi}|\cos\theta|^{1.5}\,d\theta = 3.4961$$

with $\rho$ the density [kg/m³]. Nothing new is fitted. For sinusoidal $B$, the dynamic loop area therefore equals the Bertotti total loss per cycle, $\rho\,p(f, \hat B)/f$, whenever the static Takács loop matches the hysteresis term.

The fitted $k_e$ usually differs from the physical $\sigma d^2/12$, because measured eddy loss is often larger than the classical value and the fit shares the surplus between the eddy and excess terms. `classical_eddy_coefficient(sigma, d)` gives the physical value for comparison.

**Validity: no skin effect.** The $c_{cl}$ term assumes uniform $B$ through the sheet, which needs the skin depth $\delta = \sqrt{2/(\omega\mu\sigma)}$ to exceed the half-thickness $d/2$. With $\sigma = 2\cdot10^6$ S/m and $\mu_r$ between 1000 and 5000, $\delta$ is 0.7–1.6 mm at 50 Hz but 0.16–0.36 mm at 1 kHz, against $d/2 = 0.175$ mm for 0.35 mm sheet. So the model holds at mains frequency and fails in the kHz range.

**Choosing the effects.** `field_waveform` and `cycle_energy` take two switches, so the same model can be analysed with or without each effect:

| `hysteresis` | `dynamic` | $H(t)$ | Loss per cycle |
|---|---|---|---|
| `True` | `True` | Takács loop + eddy/excess fields | static + eddy + excess |
| `True` | `False` | Takács loop | static hysteresis |
| `False` | `True` | normal curve + eddy/excess fields | eddy + excess |
| `False` | `False` | normal curve (single-valued B–H) | none |

With `hysteresis=False` the static field comes from the normal magnetisation curve of the same Takács model (`normal_H`), so all four options share the same fitted data and the same loop tips. Turning hysteresis off removes only the loop width. A single-valued curve also accepts biased $B(t)$ and minor reversals, which the loop branches reject.

**Input.** $B(t)$ is sampled at a fixed time step `dt` over one period; $\dot B$ is the periodic central difference, accurate to second order in `dt`. The basic Takács form still rejects minor reversals, so PWM ripple is not covered.

In [ ]:
#| export
_C_EXC = 3.4960767390218895  # ∫_0^{2π} |cos θ|^1.5 dθ


def classical_eddy_coefficient(sigma: float, d: float) -> float:
    """Physical classical-eddy coefficient c_cl = σ d² / 12 [A·s/(m·T)].

    Args:
        sigma: electrical conductivity of the steel [S/m]
        d: lamination thickness [m]
    """
    if sigma <= 0 or d <= 0:
        raise ValueError(f"sigma and d must be positive, got {sigma}, {d}")
    return sigma * d ** 2 / 12


@dataclass(frozen=True)
class DynamicLoopModel:
    """Static Takács loop widened by Bertotti's classical-eddy and excess fields.

    H(t) = H_static(B) + c_cl dB/dt + c_exc sign(dB/dt) |dB/dt|^0.5

    Both effects are optional per call: `hysteresis=False` replaces the loop by the
    single-valued normal curve of the same model, `dynamic=False` drops the
    rate-dependent terms.
    """
    static: TakacsModel
    c_cl: float  # [A·s/(m·T)] classical eddy coefficient
    c_exc: float  # [A/m·(s/T)^0.5] excess coefficient

    def __post_init__(self):
        if self.c_cl < 0 or self.c_exc < 0:
            raise ValueError(f"c_cl and c_exc must be >= 0, got {self.c_cl}, {self.c_exc}")

    @classmethod
    def from_bertotti(cls, static: TakacsModel, k_e: float, k_a: float, density: float):
        """Build from `fit_bertotti` coefficients k_e, k_a [W/kg] and density [kg/m³]."""
        if density <= 0:
            raise ValueError(f"density must be positive, got {density}")
        return cls(static, k_e * density / (2 * np.pi ** 2), k_a * density / (_C_EXC * np.sqrt(2 * np.pi)))

    def field_waveform(self, B, dt: float, hysteresis: bool = True, dynamic: bool = True):
        """H(t) [A/m] for one period of B(t) [T] sampled at time step dt [s].

        Args:
            hysteresis: use the Takács loop (True) or its single-valued normal curve (False).
            dynamic: add the classical-eddy and excess fields.
        """
        if dt <= 0:
            raise ValueError(f"dt must be positive, got {dt}")
        B = np.asarray(B, dtype=float)
        H = self.static.field_waveform(B) if hysteresis else self.static.normal_H(B)
        if dynamic:
            dBdt = (np.roll(B, -1) - np.roll(B, 1)) / (2 * dt)
            H = H + self.c_cl * dBdt + self.c_exc * np.sign(dBdt) * np.sqrt(np.abs(dBdt))
        return H

    def cycle_energy(self, B, dt: float, hysteresis: bool = True, dynamic: bool = True) -> float:
        """Energy per cycle ∮H dB [J/m³] for one period of B(t) sampled at dt; switches as in `field_waveform`."""
        B = np.asarray(B, dtype=float)
        H = self.field_waveform(B, dt, hysteresis, dynamic)
        return float(np.sum(0.5 * (H + np.roll(H, -1)) * (np.roll(B, -1) - B)))

In [ ]:
# Examples
# Illustrative Bertotti coefficients and density — not fitted to any real steel.
k_e, k_a, rho = 5e-5, 5e-4, 7650.0
dyn = DynamicLoopModel.from_bertotti(m, k_e, k_a, rho)
assert np.isclose(dyn.c_cl, k_e * rho / (2 * np.pi ** 2))

f, B_hat, n = 50.0, 1.3, 2000
t = np.arange(n) / (n * f)
B_t = B_hat * np.sin(2 * np.pi * f * t)
W_dyn = dyn.cycle_energy(B_t, t[1])
W_static = m.loop_energy(m.tip_H(B_hat))
W_eddy_exc = rho * (k_e * (f * B_hat) ** 2 + k_a * (f * B_hat) ** 1.5) / f   # Bertotti p_e + p_a per cycle
assert np.isclose(W_dyn, W_static + W_eddy_exc, rtol=1e-3)   # loop area = static + Bertotti dynamic loss
print(f"per cycle at {f:.0f} Hz, {B_hat} T: static {W_static:.1f} + eddy/excess {W_eddy_exc:.1f} "
      f"= {W_static + W_eddy_exc:.1f} J/m³; dynamic loop area {W_dyn:.1f} J/m³")
# the four combinations of the two switches
W = {(h, d): dyn.cycle_energy(B_t, t[1], hysteresis=h, dynamic=d) for h in (True, False) for d in (True, False)}
assert np.isclose(W[True, False], W_static, rtol=1e-3)            # static loop only
assert np.isclose(W[False, True], W_eddy_exc, rtol=1e-3)          # eddy + excess only
assert abs(W[False, False]) < 1e-9 * W_static                     # single-valued curve: no loss
# physical check: σ d²/12 for 0.35 mm sheet, σ = 2e6 S/m (typical value, not a specific steel)
print(f"c_cl from k_e = {dyn.c_cl:.4f}, physical σd²/12 = {classical_eddy_coefficient(2e6, 0.35e-3):.4f} A·s/(m·T)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
H_nc = m.normal_H(np.linspace(-B_hat, B_hat, 201))
ax.plot(H_nc, np.linspace(-B_hat, B_hat, 201), "k:", lw=1.2, label="normal curve (hysteresis off)")
for f_plot, style in [(0.0, "k--"), (50.0, "C0"), (200.0, "C3")]:
    if f_plot == 0:
        H, B = m.loop(m.tip_H(B_hat))
        ax.plot(H, B, style, lw=1.2, label="static (DC)")
        continue
    tt = np.arange(n) / (n * f_plot)
    Bt = B_hat * np.sin(2 * np.pi * f_plot * tt)
    Ht = dyn.field_waveform(Bt, tt[1])
    ax.plot(np.append(Ht, Ht[0]), np.append(Bt, Bt[0]), style, lw=1.4, label=f"{f_plot:.0f} Hz")
ax.set_xlabel("H (A/m)"); ax.set_ylabel("B (T)")
ax.set_title("Dynamic widening, illustrative coefficients")
ax.grid(True, alpha=0.4); ax.legend(fontsize=8)
plt.show()

## Tests

In [ ]:
# loop energy: closed form equals the numerical integral of descending minus ascending branch
for xm in [0.5, 1.0, 2.5, 6.0]:
    H_m = xm * m.h0
    num = quad(lambda H: m.branch_B(H, H_m, False) - m.branch_B(H, H_m, True), -H_m, H_m)[0]
    assert np.isclose(m.loop_energy(H_m), num, rtol=1e-9), xm
assert np.isfinite(m.loop_energy(1e6 * m.h0))                  # no overflow far into saturation

# inverse branches: B -> H -> B round trip on both branches
B_m = 1.2
B = np.linspace(-B_m, B_m, 101)
H_tipB = m.tip_H(B_m)
for rising in (True, False):
    assert np.allclose(m.branch_B(m.branch_H(B, B_m, rising), H_tipB, rising), B)

# normal curve is monotonic and below B_s (range kept where tanh is still resolvable in float64)
H = np.linspace(1, 8 * m.h0, 500)
assert np.all(np.diff(m.tip_B(H)) > 0) and m.tip_B(H).max() < m.B_s

# invalid inputs
for bad in [lambda: TakacsModel(1.6, 50, 0), lambda: m.tip_H(1.6), lambda: m.tip_H(0),
            lambda: m.loop_energy(0), lambda: m.field_waveform(1.3 * np.sin(theta) + 0.2),
            lambda: m.field_waveform(np.sin(theta) + 0.3 * np.sin(3 * theta)),   # symmetric, but with minor reversals
            lambda: fit_takacs([1, 2], [0.1, 0.2], [], [])]:
    try:
        bad()
    except ValueError:
        pass
    else:
        raise AssertionError("no ValueError")
print("✓ Takács model tests passed")

In [ ]:
# dynamic terms vanish without coefficients: same H(t) as the static model
dyn0 = DynamicLoopModel(m, 0.0, 0.0)
assert np.allclose(dyn0.field_waveform(B_t, t[1]), m.field_waveform(B_t))
# switches: dynamic=False gives the static loop, hysteresis=False the normal curve
assert np.allclose(dyn.field_waveform(B_t, t[1], dynamic=False), m.field_waveform(B_t))
assert np.allclose(dyn.field_waveform(B_t, t[1], hysteresis=False, dynamic=False), m.normal_H(B_t))
# hysteresis off and dynamic on: the field is the normal curve plus the same dynamic terms
H_dyn_only = dyn.field_waveform(B_t, t[1], hysteresis=False) - m.normal_H(B_t)
assert np.allclose(H_dyn_only, dyn.field_waveform(B_t, t[1]) - m.field_waveform(B_t))
# a single-valued curve accepts biased B(t) and minor reversals, which the loop rejects
B_bias = 0.5 + 0.6 * np.sin(2 * np.pi * f * t)
assert np.all(np.isfinite(dyn.field_waveform(B_bias, t[1], hysteresis=False)))
# normal curve is the odd inverse of the tip curve
H_chk = np.array([30.0, 80.0, 200.0])
assert np.allclose(m.normal_H(m.tip_B(H_chk)), H_chk) and np.allclose(m.normal_H(-m.tip_B(H_chk)), -H_chk)
# the tip does not move: dB/dt = 0 at the peak of B
i_pk = np.argmax(B_t)
assert np.isclose(dyn.field_waveform(B_t, t[1])[i_pk], m.field_waveform(B_t)[i_pk], rtol=1e-3)
# cycle energy of each term alone matches its Bertotti loss (eddy ∝ f², excess ∝ f^1.5)
for f_chk in (50.0, 400.0):
    tt = np.arange(n) / (n * f_chk)
    Bt = B_hat * np.sin(2 * np.pi * f_chk * tt)
    W_s = m.loop_energy(m.tip_H(B_hat))
    W_e = DynamicLoopModel.from_bertotti(m, k_e, 0.0, rho).cycle_energy(Bt, tt[1]) - W_s
    W_a = DynamicLoopModel.from_bertotti(m, 0.0, k_a, rho).cycle_energy(Bt, tt[1]) - W_s
    assert np.isclose(W_e, rho * k_e * (f_chk * B_hat) ** 2 / f_chk, rtol=1e-3), f_chk
    assert np.isclose(W_a, rho * k_a * (f_chk * B_hat) ** 1.5 / f_chk, rtol=1e-3), f_chk
assert np.isclose(classical_eddy_coefficient(2e6, 0.35e-3), 2e6 * 0.35e-3 ** 2 / 12)
for bad in [lambda: DynamicLoopModel(m, -1.0, 0.0), lambda: dyn.field_waveform(B_t, 0.0),
            lambda: DynamicLoopModel.from_bertotti(m, k_e, k_a, 0.0), lambda: classical_eddy_coefficient(0, 1e-3)]:
    try:
        bad()
    except ValueError:
        pass
    else:
        raise AssertionError("no ValueError")
print("✓ Dynamic widening tests passed")

## References

- Takács, J. (2001). A phenomenological mathematical model of hysteresis. *COMPEL*, 20(4), 1002–1015.
- Takács, J. (2003). *Mathematics of Hysteretic Phenomena: The T(x) Model for the Description of Hysteresis*. Wiley-VCH.
- Bertotti, G. (1988). General properties of power losses in soft ferromagnetic materials. *IEEE Transactions on Magnetics*, 24(1), 621–630 — loss and field separation.
- Bertotti, G. (1998). *Hysteresis in Magnetism*. Academic Press — loss separation used for the loop energy.